# Ekstrakcja cech

In [2]:
import pandas as pd
import librosa
from pyprojroot import here
from tqdm.notebook import tqdm

In [3]:
ROOT = here()
AUDIO_DIR = ROOT / "ESC-50" / "audio"
METADATA_PATH = ROOT / "ESC-50" / "meta" / "esc50.csv"

In [ ]:
def add_coeff_stats(features_dict, array_2d, name):
    mean_vals = array_2d.mean(axis=1)
    std_vals = array_2d.std(axis=1)
    for i, (m, s) in enumerate(zip(mean_vals, std_vals)):
        features_dict[f'{name}_{i+1}_mean'] = m
        features_dict[f'{name}_{i+1}_std'] = s

In [ ]:
def extract_features(file_path, sr=22050):
    y, _ = librosa.load(file_path, sr=sr, duration=5)

    features = {}

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    add_coeff_stats(features, mfcc, 'mfcc')
    
    delta_mfcc = librosa.feature.delta(mfcc)
    add_coeff_stats(features, delta_mfcc, 'delta_mfcc')

    delta_mfcc2 = librosa.feature.delta(mfcc, order=2)
    add_coeff_stats(features, delta_mfcc2, 'delta_mfcc2')

    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    add_coeff_stats(features, chroma, 'chroma')

    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    add_coeff_stats(features, contrast, 'contrast')

    tonnetz = librosa.feature.tonnetz(y=librosa.effects.harmonic(y), sr=sr)
    add_coeff_stats(features, tonnetz, 'tonnetz')

    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    features['spectral_centroid_mean'] = centroid[0].mean()
    features['spectral_centroid_std']  = centroid[0].std()

    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    features['spectral_rolloff_mean']  = rolloff[0].mean()
    features['spectral_rolloff_std']   = rolloff[0].std()

    flatness = librosa.feature.spectral_flatness(y=y)
    features['spectral_flatness_mean'] = flatness[0].mean()
    features['spectral_flatness_std']  = flatness[0].std()

    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    features['spectral_bandwidth_mean']= bandwidth[0].mean()
    features['spectral_bandwidth_std'] = bandwidth[0].std()

    zcr = librosa.feature.zero_crossing_rate(y)
    features['zcr_mean'] = zcr[0].mean()
    features['zcr_std']  = zcr[0].std()

    rms = librosa.feature.rms(y=y)
    features['rms_mean'] = rms[0].mean()
    features['rms_std']  = rms[0].std()

    return features

In [ ]:
meta = pd.read_csv(METADATA_PATH)
feature_list = []

for idx, row in tqdm(meta.iterrows(), total=len(meta)):
    file_path = AUDIO_DIR / row['filename']
    features = extract_features(file_path)
    features['target']   = row['target']
    features['category'] = row['category']
    features['fold']     = row['fold']

    feature_list.append(features)

features_df = pd.DataFrame(feature_list)
features_df.to_csv(ROOT / "data" / "esc50_features.csv", index=False)

In [4]:
df = pd.read_csv(ROOT / "data" / "esc50_features.csv")
print(df.shape)
df.head()

(2000, 185)


,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,mfcc_3_mean,mfcc_3_std,mfcc_4_mean,mfcc_4_std,mfcc_5_mean,mfcc_5_std,...,spectral_flatness_std,spectral_bandwidth_mean,spectral_bandwidth_std,zcr_mean,zcr_std,rms_mean,rms_std,target,category,fold
0,-600.940200,107.966120,4.694290,17.719593,-8.505940,30.912415,-4.276621,15.667683,-0.818615,4.175005,...,0.273920,177.267664,571.589715,0.013450,0.043230,0.008202,0.040638,0,dog,1
1,-193.978200,21.637465,2.746146,19.263046,-59.386883,14.980329,4.413459,8.313001,-47.953037,11.499779,...,0.027195,2255.062496,106.181423,0.303512,0.041146,0.049820,0.035493,14,chirping_birds,1
2,16.259266,18.007887,56.921246,9.777822,-8.960625,4.349467,15.122986,4.913087,-12.898559,3.722227,...,0.014455,2756.101423,86.164400,0.385365,0.060474,0.270203,0.045551,36,vacuum_cleaner,1
3,17.771095,23.531942,54.864986,3.419258,-7.179326,4.562182,13.654805,3.863994,-9.484007,3.711377,...,0.015245,2837.942479,72.646173,0.386870,0.044447,0.274092,0.064845,36,vacuum_cleaner,1
4,-423.075230,20.478466,124.929820,21.618866,38.760685,10.372663,35.040855,7.883809,-2.481526,8.798615,...,0.004883,2209.732054,343.443920,0.045898,0.013067,0.008937,0.006286,19,thunderstorm,1
